# FIFA Player Analysis
Navya Sharma

This notebook explores the FIFA 22 Complete Player Dataset (Kaggle, by Stefano Leone).
Dataset link: https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset

Goal: analyze players, clubs, countries, overall ratings, market value, wages and positions,
then export clean summary tables that will be pulled into Tableau for the dashboard.


In [ ]:
# Step 1: Get the dataset onto Colab
# Option A (recommended): upload your kaggle.json API key, then run the kaggle CLI
# Option B: manually download players_22.csv from the Kaggle link above and upload it
#           using the file icon on the left sidebar of Colab.

# ---- Option A: Kaggle API ----
# from google.colab import files
# files.upload()  # upload kaggle.json here
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d stefanoleone992/fifa-22-complete-player-dataset
# !unzip -o fifa-22-complete-player-dataset.zip

# ---- Option B: manual upload ----
from google.colab import files
uploaded = files.upload()  # choose players_22.csv from your machine


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("players_22.csv", low_memory=False)
print(df.shape)
df.head()


## 1. Cleaning up the columns we actually need
The raw file has 100+ attributes. We only keep what's relevant for player, club, country, rating, value, wage and position analysis.

In [ ]:
cols = [
    "short_name", "long_name", "age", "nationality_name", "club_name", "league_name",
    "overall", "potential", "value_eur", "wage_eur", "player_positions",
    "preferred_foot", "international_reputation", "weak_foot", "skill_moves",
    "height_cm", "weight_kg", "club_position", "club_jersey_number"
]
fifa = df[cols].copy()

# basic cleaning
fifa["value_eur"] = fifa["value_eur"].fillna(0)
fifa["wage_eur"] = fifa["wage_eur"].fillna(0)
fifa["main_position"] = fifa["player_positions"].str.split(",").str[0]
fifa = fifa.dropna(subset=["overall", "club_name", "nationality_name"])
fifa.info()


## 2. Overall rating distribution

In [ ]:
plt.hist(fifa["overall"], bins=30, color="#1f77b4", edgecolor="white")
plt.title("Distribution of Player Overall Ratings")
plt.xlabel("Overall Rating")
plt.ylabel("Number of Players")
plt.show()

fifa["overall"].describe()


## 3. Top countries by number of players and by average rating

In [ ]:
top_countries_count = fifa["nationality_name"].value_counts().head(15)
top_countries_count.plot(kind="barh", color="#2ca02c")
plt.title("Top 15 Countries by Number of Players")
plt.xlabel("Player Count")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
avg_rating_country = (fifa.groupby("nationality_name")["overall"]
                          .agg(["mean", "count"])
                          .query("count >= 50")
                          .sort_values("mean", ascending=False)
                          .head(15))
avg_rating_country


## 4. Top clubs by average squad rating (min. 15 players)

In [ ]:
club_strength = (fifa.groupby("club_name")["overall"]
                    .agg(["mean", "count"])
                    .query("count >= 15")
                    .sort_values("mean", ascending=False)
                    .head(15))
club_strength.rename(columns={"mean": "avg_overall", "count": "squad_size"}, inplace=True)
club_strength


## 5. Market value and wages

In [ ]:
top_value = fifa.sort_values("value_eur", ascending=False)[
    ["short_name", "club_name", "nationality_name", "overall", "value_eur", "wage_eur"]
].head(15)
top_value


In [ ]:
plt.scatter(fifa["overall"], fifa["value_eur"], alpha=0.3, s=10)
plt.title("Overall Rating vs Market Value")
plt.xlabel("Overall Rating")
plt.ylabel("Value (EUR)")
plt.yscale("log")
plt.show()


## 6. Positions breakdown

In [ ]:
position_counts = fifa["main_position"].value_counts()
position_counts.plot(kind="bar", color="#ff7f0e")
plt.title("Player Count by Main Position")
plt.xlabel("Position")
plt.ylabel("Count")
plt.show()


In [ ]:
avg_rating_by_position = fifa.groupby("main_position")["overall"].mean().sort_values(ascending=False)
avg_rating_by_position


## 7. Age vs rating (are players peaking at a certain age?)

In [ ]:
age_curve = fifa.groupby("age")["overall"].mean()
age_curve.plot(marker="o")
plt.title("Average Overall Rating by Age")
plt.xlabel("Age")
plt.ylabel("Average Overall")
plt.show()


## 8. Export clean tables for Tableau
Tableau will connect to these CSVs (or you can publish them straight into Tableau Public / Tableau Desktop).
Keep this cell exactly as-is if you want the same field names used in the Flask app's Tableau embed later.


In [ ]:
fifa.to_csv("fifa_players_clean.csv", index=False)
club_strength.to_csv("club_strength_summary.csv")
avg_rating_country.to_csv("country_rating_summary.csv")
position_counts.to_csv("position_counts.csv")

from google.colab import files
files.download("fifa_players_clean.csv")
files.download("club_strength_summary.csv")
files.download("country_rating_summary.csv")
files.download("position_counts.csv")


## 9. Building the Tableau dashboard (manual step, outside Colab)
1. Open Tableau Desktop (or Tableau Public — it's free).
2. Connect to `fifa_players_clean.csv`.
3. Build sheets: rating distribution, top clubs, top countries, value vs rating scatter, position breakdown.
4. Combine them into one Dashboard.
5. Publish to **Tableau Public** (Server > Save to Tableau Public).
6. Copy the shareable/embed link from Tableau Public — this is what goes into `app.py` as `TABLEAU_VIEW_URL`.
